<a href="https://colab.research.google.com/github/GabyPugaBR/AAI2025/blob/main/1_Basic_ReAct_Agent_in_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install tenacity==9.0.0
%pip install 'pydantic<2'
%pip install langchain==0.3.12
%pip install langchain-openai==0.2.12
%pip install langchain_community==0.3.12
%pip install langgraph==0.2.59
%pip install pysqlite3-binary==0.5.4
%pip install langchain_chroma==0.1.4
%pip install pandas==2.2.3
%pip install pypdf==5.1.0
%pip install nbformat==5.10.4

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.26
    Uninstalling pydantic-1.10.26:
      Successfully uninstalled pydantic-1.10.26
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.3.63 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.3.8 which is incomp

## Setup Function Tools for ReAct Agent

In [2]:
from langchain_core.tools import tool

#Tool annotation identifies a function as a tool automatically
@tool
def find_sum(x:int, y:int) -> int :
    #The docstring comment describes the capabilities of the function
    #It is used by the agent to discover the function's inputs, outputs and capabilities
    """
    This function is used to add two numbers and return their sum.
    It takes two integers as inputs and returns an integer as output.
    """
    return x + y

@tool
def find_product(x:int, y:int) -> int :
    """
    This function is used to multiply two numbers and return their product.
    It takes two integers as inputs and returns an integer as ouput.
    """
    return x * y


## Create a basic ReAct Agent

In [3]:
from langchain_openai import ChatOpenAI
import os
from google.colab import userdata

model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=userdata.get("OpenAI_API")
)
#Test the model
response = model.invoke("Hello, how are you?")
print(response.content)


Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


In [14]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage

#Create list of tools available to the agent
agent_tools=[find_sum, find_product]

#System prompt
system_prompt = SystemMessage(
    """You are a Math genius who can solve math problems. Solve the
    problems provided by the user, by using only tools available.
    Do not solve the problem yourself"""
)

agent_graph=create_react_agent(
    model=model,
    state_modifier=system_prompt,
    tools=agent_tools)


## Execute the ReAct Agent

In [15]:
#Example 1
inputs = {"messages":[("user","what is the sum of 2 and 3 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

Agent returned : The sum of 2 and 3 is 5. 

Step by Step execution : 
================================ Human Message =================================

what is the sum of 2 and 3 ?
================================== Ai Message ==================================
Tool Calls:
  find_sum (call_XEEXs4HU7gNT3CX4e1ffVC94)
 Call ID: call_XEEXs4HU7gNT3CX4e1ffVC94
  Args:
    x: 2
    y: 3
================================= Tool Message =================================
Name: find_sum

5
================================== Ai Message ==================================

The sum of 2 and 3 is 5.


In [16]:
#Example 2
inputs = {"messages":[("user","What is 3 multipled by 2 and 5 + 1 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

Agent returned : The result of 3 multiplied by 2 is 6, and the result of 5 plus 1 is also 6. 

Step by Step execution : 
================================ Human Message =================================

What is 3 multipled by 2 and 5 + 1 ?
================================== Ai Message ==================================
Tool Calls:
  find_product (call_he8xyE0d9sNRr4Tba9SPAPPR)
 Call ID: call_he8xyE0d9sNRr4Tba9SPAPPR
  Args:
    x: 3
    y: 2
  find_sum (call_ckXpOGZNrG6mvZBw8oOGte9P)
 Call ID: call_ckXpOGZNrG6mvZBw8oOGte9P
  Args:
    x: 5
    y: 1
================================= Tool Message =================================
Name: find_product

6
================================= Tool Message =================================
Name: find_sum

6
================================== Ai Message ==================================

The result of 3 multiplied by 2 is 6, and the result of 5 plus 1 is also 6.


## Debugging the Agent

In [17]:
agent_graph=create_react_agent(
    model=model,
    state_modifier=system_prompt,
    tools=agent_tools,
    debug=True)

inputs = {"messages":[("user","what is the sum of 2 and 3 ?")]}

result = agent_graph.invoke(inputs)

[-1:checkpoint] State at the end of step -1:
{'messages': []}
[0:tasks] Starting 1 task for step 0:
- __start__ -> {'messages': [('user', 'what is the sum of 2 and 3 ?')]}
[0:writes] Finished step 0 with writes to 1 channel:
- messages -> [('user', 'what is the sum of 2 and 3 ?')]
[0:checkpoint] State at the end of step 0:
{'messages': [HumanMessage(content='what is the sum of 2 and 3 ?', additional_kwargs={}, response_metadata={}, id='a3bcdf8e-6886-43bd-88b5-fc2f4080287f')]}
[1:tasks] Starting 1 task for step 1:
- agent -> {'is_last_step': False,
 'messages': [HumanMessage(content='what is the sum of 2 and 3 ?', additional_kwargs={}, response_metadata={}, id='a3bcdf8e-6886-43bd-88b5-fc2f4080287f')],
 'remaining_steps': 24}
[1:writes] Finished step 1 with writes to 1 channel:
- messages -> [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_JHThv27Tb1LVucdR7FUZPJy4', 'function': {'arguments': '{"x":2,"y":3}', 'name': 'find_sum'}, 'type': 'function'}], 'refusal': None}